# Loan Approval Prediction Description

**Goal:** predict whether a loan application is approved, and evaluate it properly on imbalanced data (precision, recall, F1).

**Plan**
1. Load and clean the data
2. Explore (EDA)
3. Stratified train/test split
4. Preprocessing pipeline (missing values + categorical encoding)
5. Models: Logistic Regression vs Decision Tree (bonus), with class weights and SMOTE (bonus)
6. Cross-validated comparison, tuning, final test evaluation
7. Decision threshold tuning, feature importance
8. Conclusions and limitations

**Dataset:** Loan-Approval-Prediction-Dataset (Kaggle). Download the CSV and set `DATA_PATH` below.
The notebook auto-handles common quirks (spaces in column names/values, ID column), so it also works with similar loan datasets.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay, precision_recall_curve,
                             average_precision_score, roc_auc_score, precision_score, recall_score, f1_score)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)

In [ ]:
# SMOTE lives in imbalanced-learn. Its Pipeline applies SMOTE ONLY when fitting (never to validation/test data).
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
except ImportError:
    %pip install imbalanced-learn
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline

## 1. Load and clean

Typical problems in this dataset: column names and string values with leading/trailing spaces (e.g. `" loan_status"`, `" Approved"`), and an ID column that must not be used as a feature.

In [ ]:
DATA_PATH = "loan_approval_dataset.csv"   # <-- change to your file path

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
for c in df.select_dtypes(include=["object", "string"]).columns:
    df[c] = df[c].str.strip()

print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
df.head()

In [ ]:
# Identify target and ID columns automatically
TARGET = next(c for c in df.columns if c.lower() == "loan_status")
ID_COLS = [c for c in df.columns if c.lower() in ("loan_id", "id")]

print("Target:", TARGET, "| ID columns dropped:", ID_COLS)
print(df[TARGET].value_counts(dropna=False))

# Positive class (1) = Approved
POSITIVE_VALUES = {"approved", "y", "yes", "1"}
y = df[TARGET].astype(str).str.lower().isin(POSITIVE_VALUES).astype(int)
X = df.drop(columns=[TARGET] + ID_COLS)

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
print("\nNumeric:", num_cols)
print("Categorical:", cat_cols)

## 2. Exploratory data analysis

Questions to answer here (write your findings in the cell below the plots):
- How imbalanced is the target?
- Are there missing values? (Some versions of this dataset have none. We still build imputation into the pipeline, because real data will have gaps.)
- Which features separate approved from rejected?

In [ ]:
print("Missing values per column:")
print(X.isna().sum()[X.isna().sum() > 0] if X.isna().any().any() else "None")

dist = y.value_counts().rename({1: "Approved", 0: "Rejected"})
print("\nClass balance:\n", pd.DataFrame({"count": dist, "share": (dist / len(y)).round(3)}))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
dist.plot(kind="bar", ax=axes[0], color=["#4C72B0", "#C44E52"]); axes[0].set_title("Target distribution")
axes[0].tick_params(axis="x", rotation=0)
X.assign(target=y).corr(numeric_only=True)["target"].drop("target").sort_values().plot(kind="barh", ax=axes[1])
axes[1].set_title("Correlation with approval (numeric features)")
plt.tight_layout(); plt.show()

X[num_cols].hist(bins=25, figsize=(12, 7)); plt.tight_layout(); plt.show()

for c in cat_cols:
    print(f"\nApproval rate by {c}:")
    print(y.groupby(X[c]).mean().round(3))

**My EDA notes** (fill in): class balance is ..., strongest signals are ..., data quality issues are ...

## 3. Train/test split

Stratified, so train and test keep the same class ratio. The test set stays untouched until the final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)
print("Approved share  train:", y_train.mean().round(3), " test:", y_test.mean().round(3))

## 4. Preprocessing and model pipelines

Everything (imputation, scaling, encoding, SMOTE) lives **inside** the pipeline. That way, during cross-validation each fold learns its preprocessing only from its own training portion, so no leakage.

- Numeric: median imputation, then standard scaling (matters for logistic regression)
- Categorical: most-frequent imputation, then one-hot encoding (unknown categories ignored)

In [ ]:
def make_preprocessor():
    num = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
    cat = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                    ("oh", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", num, num_cols), ("cat", cat, cat_cols)])

def build(model, smote=False):
    steps = [("prep", make_preprocessor())]
    if smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
    steps.append(("model", model))
    return ImbPipeline(steps)

LR = lambda **kw: LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, **kw)
DT = lambda **kw: DecisionTreeClassifier(random_state=RANDOM_STATE, **kw)

experiments = {
    "Dummy (always majority)": build(DummyClassifier(strategy="most_frequent")),
    "LogReg":                  build(LR()),
    "LogReg + class_weight":   build(LR(class_weight="balanced")),
    "LogReg + SMOTE":          build(LR(), smote=True),
    "Tree":                    build(DT()),
    "Tree + class_weight":     build(DT(class_weight="balanced")),
    "Tree + SMOTE":            build(DT(), smote=True),
}

## 5. Cross-validated comparison (training data only)

5-fold stratified CV. We track precision, recall, F1 for the *Approved* class plus ROC-AUC and PR-AUC.
The **Dummy** row is your reality check: a model that always says "Approved" already gets high accuracy on imbalanced data, which is why accuracy is not the headline metric here.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"precision": "precision", "recall": "recall", "f1": "f1",
           "roc_auc": "roc_auc", "pr_auc": "average_precision"}

rows = []
for name, pipe in experiments.items():
    s = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({"model": name, **{k: s[f"test_{k}"].mean() for k in scoring}})

cv_table = pd.DataFrame(rows).set_index("model").round(3).sort_values("f1", ascending=False)
cv_table

**Read the table:** Does SMOTE or class weighting actually help here, or is the data only mildly imbalanced so the gain is tiny? Report what you see honestly, even if "no big difference" is the answer.

## 6. Hyperparameter tuning (Decision Tree)

An unconstrained tree memorizes the training data. We limit depth and leaf size and pick by cross-validated F1.

In [ ]:
param_grid = {
    "model__max_depth": [3, 4, 5, 6, 8, None],
    "model__min_samples_leaf": [1, 5, 10, 20],
    "model__criterion": ["gini", "entropy"],
}
gs = GridSearchCV(build(DT()), param_grid, scoring="f1", cv=cv, n_jobs=-1)
gs.fit(X_train, y_train)
print("Best params:", gs.best_params_)
print("Best CV F1:", round(gs.best_score_, 3))

## 7. Final evaluation on the held-out test set

We evaluate a small set of finalists once, at the default 0.5 threshold. Never choose models by looking at test results; that is what Section 5 and 6 are for.

In [ ]:
def evaluate(name, pipe, X_te, y_te, threshold=0.5, show=True):
    proba = pipe.predict_proba(X_te)[:, 1]
    pred = (proba >= threshold).astype(int)
    if show:
        print(f"===== {name} (threshold={threshold:.2f}) =====")
        print(classification_report(y_te, pred, target_names=["Rejected", "Approved"], digits=3))
        ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=["Rejected", "Approved"], cmap="Blues")
        plt.title(name); plt.show()
    return {"model": name, "threshold": round(threshold, 3),
            "precision": precision_score(y_te, pred), "recall": recall_score(y_te, pred),
            "f1": f1_score(y_te, pred), "roc_auc": roc_auc_score(y_te, proba),
            "pr_auc": average_precision_score(y_te, proba)}

finalists = {
    "LogReg + class_weight": build(LR(class_weight="balanced")),
    "Decision Tree (tuned)": gs.best_estimator_,
}
test_rows = []
for name, pipe in finalists.items():
    if name != "Decision Tree (tuned)":   # GridSearchCV already refit the best tree on all training data
        pipe.fit(X_train, y_train)
    test_rows.append(evaluate(name, pipe, X_test, y_test))

pd.DataFrame(test_rows).set_index("model").round(3)

## 8. Decision threshold tuning

The default 0.5 cutoff is arbitrary. For a bank, **approving a loan that should be rejected is usually the costly mistake**, which means we care about the *precision* of the Approved class.

We pick the threshold using **out-of-fold predictions on the training data** (not the test set), requiring a target precision, then maximizing recall under that constraint.

In [ ]:
final_model = build(LR(class_weight="balanced"))

oof_proba = cross_val_predict(final_model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
prec, rec, thr = precision_recall_curve(y_train, oof_proba)

TARGET_PRECISION = 0.95   # <-- business decision: adjust and discuss
ok = prec[:-1] >= TARGET_PRECISION
best_thr = thr[ok][np.argmax(rec[:-1][ok])] if ok.any() else 0.5
print(f"Chosen threshold: {best_thr:.3f}  (target precision >= {TARGET_PRECISION})")

plt.figure(figsize=(6, 4))
plt.plot(rec, prec); plt.axhline(TARGET_PRECISION, ls="--", c="gray")
plt.xlabel("Recall (Approved)"); plt.ylabel("Precision (Approved)"); plt.title("Precision-Recall curve (out-of-fold)")
plt.show()

final_model.fit(X_train, y_train)
tuned_row = evaluate("LogReg + class_weight, tuned threshold", final_model, X_test, y_test, threshold=best_thr)
pd.DataFrame(test_rows + [tuned_row]).set_index("model").round(3)

## 9. What drives the decisions? (feature importance)

Two views: the tree's built-in importance (fast, but biased toward high-cardinality features) and **permutation importance** on the test set (model-agnostic, more trustworthy).

In [ ]:
best_tree = gs.best_estimator_
names = best_tree.named_steps["prep"].get_feature_names_out()
tree_imp = pd.Series(best_tree.named_steps["model"].feature_importances_, index=names).sort_values(ascending=False).head(10)

perm = permutation_importance(final_model, X_test, y_test, scoring="f1", n_repeats=15,
                              random_state=RANDOM_STATE, n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
tree_imp[::-1].plot(kind="barh", ax=axes[0]); axes[0].set_title("Decision tree importance (top 10)")
perm_imp[::-1].plot(kind="barh", ax=axes[1]); axes[1].set_title("Permutation importance, LogReg (drop in F1)")
plt.tight_layout(); plt.show()

## 10. Conclusions and limitations (write your own version)

**Results:** best model is ..., with precision ..., recall ..., F1 ... for the Approved class on the test set.

**Logistic Regression vs Decision Tree:** ...

**Did imbalance handling help?** (class weights / SMOTE / threshold tuning): ...

**Limitations to mention**
- Dataset is small and likely synthetic or simplified, so scores may not transfer to real lending.
- A very dominant feature (such as credit score) can make the problem look easier than it is. Check what happens when you remove it.
- Fairness: real credit models must be audited for bias, and this analysis does not do that.
- Metrics depend on the chosen threshold, which should come from actual business costs.

**Next steps:** try gradient boosting, calibrate probabilities, add a cost-based threshold, check subgroup performance.